# LLaVA-Video Inference on SUTD Traffic Video QA

This notebook runs LLaVA-Video-7B model for traffic video question answering on vast.ai.

## Features:
- 4-bit/8-bit quantization support for reduced VRAM usage
- Optimized prompts for traffic safety analysis
- Compatible with SUTD Traffic Video QA dataset

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:

# Install dependencies for LLaVA-Video
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install transformers==4.55.4 accelerate==1.11.0
!pip install -U bitsandbytes  # For quantization
!pip install flash-attn --no-build-isolation --no-cache-dir
!pip install av numpy pillow  # PyAV for video processing
!pip install einops  # Required by LLaVA
!pip install open-clip-torch  # OpenCLIP
!pip install pandas tqdm loguru
!pip install kaggle

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 615.6 kB/s eta 0:00:000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 4.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.1
    Uninstalling transformers-4.40.1:
      Successfully uninstalled transformers-4.40.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0


In [35]:
!pip show transformers accelerate

Name: transformers
Version: 4.40.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: 
---
Name: accelerate
Version: 1.12.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The HuggingFace team
Author-email: zach.mueller@huggingface.co
License: Apache
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: 


In [3]:
# Install LLaVA from source
!pip install git+https://github.com/LLaVA-VL/LLaVA-NeXT.git

  Cloning https://github.com/LLaVA-VL/LLaVA-NeXT.git to /tmp/pip-req-build-20f5s_71
  Running command git clone --filter=blob:none --quiet https://github.com/LLaVA-VL/LLaVA-NeXT.git /tmp/pip-req-build-20f5s_71
  Resolved https://github.com/LLaVA-VL/LLaVA-NeXT.git to commit e9835311c6f515a13702eb7a7750fcd936f65ed8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for llava: filename=llava-1.7.0.dev0-py3-none-any.whl size=467240 sha256=37eea4b63721ff4712788971c3b069ce6bd3ce9531a7b4bb11504239db3e1147
  Stored in directory: /tmp/pip-ephem-wheel-cache-408_v925/wheels/6f/e8/96/5ef1eb145c04eab181f7e192feb9b10433ce016ed51959ca2f
Successfully built llava


In [ ]:
!pip install flash-attn --no-build-isolation --no-cache-dir

In [4]:
# Setup Kaggle credentials and download dataset
!rm -rf ~/.kaggle
!mkdir -p ~/.kaggle
!echo '{"username":"YOUR_USERNAME","key":"YOUR_KAGGLE_KEY"}' > ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download ohonsam/sutd-traffic-video-qa
!mkdir -p /kaggle/input/sutd-traffic-video-qa
!unzip -o sutd-traffic-video-qa.zip -d /kaggle/input/sutd-traffic-video-qa

Dataset URL: https://www.kaggle.com/datasets/ohonsam/sutd-traffic-video-qa
License(s): CC0-1.0
100%|██████████████████████████████████████| 37.2G/37.2G [19:22<00:00, 39.7MB/s]
100%|██████████████████████████████████████| 37.2G/37.2G [19:22<00:00, 34.3MB/s]
Archive:  sutd-traffic-video-qa.zip
  inflating: /kaggle/input/sutd-traffic-video-qa/configs/configs/config.yaml  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_all.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.txt  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_train.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_train_augmented.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_all.jsonl  
  inflating: /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_test.jsonl  
  inflating: /kaggle/inpu

In [5]:
!rm -rf sutd-traffic-video-qa.zip

In [22]:
%%writefile setup_paths.py
import os
import glob

DATA_ROOT = "/kaggle/input/sutd-traffic-video-qa"
VIDEO_DIR = f"{DATA_ROOT}/videos_obj_tracking"  # or videos_obj_tracking for YOLO tracked videos
QUESTIONS_DIR = f"{DATA_ROOT}/questions/questions"

print("VIDEO_DIR:", VIDEO_DIR, "exists:", os.path.isdir(VIDEO_DIR))
print("QUESTIONS_DIR:", QUESTIONS_DIR, "exists:", os.path.isdir(QUESTIONS_DIR))

# Find question jsonl files
question_jsons = sorted(glob.glob(os.path.join(QUESTIONS_DIR, "**", "*.jsonl"), recursive=True))
print(f"Found {len(question_jsons)} question JSONL file(s):")
for p in question_jsons[:50]:
    print(" -", p)
if len(question_jsons) > 50:
    print(" ...")

# Pick split
MODE = "test"  # change to "train" / "val" if needed

def pick_questions_path(split: str) -> str:
    split = (split or "").lower()
    for p in question_jsons:
        if split and split in os.path.basename(p).lower():
            return p
    return question_jsons[0] if question_jsons else ""

QUESTIONS_PATH = pick_questions_path(MODE)
print("QUESTIONS_PATH:", QUESTIONS_PATH)

OUTPUT_DIR = "/kaggle/working/output"
KEYFRAME_DIR = "/kaggle/temp/keyframes"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(KEYFRAME_DIR, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("KEYFRAME_DIR:", KEYFRAME_DIR)

Overwriting setup_paths.py


In [28]:
%%writefile llava_video_sutd.py
import torch
import json
import os
import gc
import copy
import numpy as np
import pandas as pd
import av
import warnings
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from transformers import BitsAndBytesConfig
import torchvision.transforms as T

warnings.filterwarnings("ignore")

# Import LLaVA components
from llava.model.builder import load_pretrained_model
from llava.mm_utils import tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from llava.conversation import conv_templates

from setup_paths import VIDEO_DIR, QUESTIONS_PATH, OUTPUT_DIR, KEYFRAME_DIR

# =============================================================================
# CONFIGURATION
# =============================================================================

QUANTIZATION = "4bit"  # Options: "4bit", "8bit", or None for full precision
NUM_FRAMES = 8  # Number of frames to extract per video
DEVICE = "cuda:0"

# =============================================================================
# MODEL LOADING (Singleton Pattern)
# =============================================================================

_model = None
_tokenizer = None
_image_processor = None
_current_device = None


def get_dashcam_augmentation():
    """Data augmentation for dashcam footage."""
    return T.Compose([
        T.ToTensor(),
        T.ColorJitter(
            brightness=0.3,
            contrast=0.3,
            saturation=0.2,
            hue=0.1
        ),
        T.RandomAdjustSharpness(sharpness_factor=1.5, p=0.3),
        T.RandomAutocontrast(p=0.2),
        T.ToPILImage(),
    ])

def load_model(device: str = "cuda:0", quantization: str = "4bit"):
    """Load the LLaVA-Video model with quantization support."""
    global _model, _tokenizer, _image_processor, _current_device

    if _model is None or _current_device != device:
        if _model is not None:
            del _model
            gc.collect()
            torch.cuda.empty_cache()

        print(f"Loading LLaVA-Video model with {quantization} quantization...")
        pretrained = "lmms-lab/LLaVA-Video-7B-Qwen2"
        model_name = "llava_qwen"

        # Configure quantization
        if quantization == "4bit":
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )
            torch_dtype = torch.float16
        elif quantization == "8bit":
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_threshold=6.0,
            )
            torch_dtype = torch.float16
        else:
            quantization_config = None
            torch_dtype = torch.bfloat16

        # For quantized models, don't pass device_map - let bitsandbytes handle it
        if quantization in ["4bit", "8bit"]:
            _tokenizer, _model, _image_processor, max_length = load_pretrained_model(
                pretrained,
                None,
                model_name,
                torch_dtype=torch_dtype,
                quantization_config=quantization_config,
            )
        else:
            _tokenizer, _model, _image_processor, max_length = load_pretrained_model(
                pretrained,
                None,
                model_name,
                torch_dtype=torch_dtype,
                device_map="auto",
            )

        _model.eval()
        _current_device = device
        print(f"VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print(f"Peak VRAM usage: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

    return _tokenizer, _model, _image_processor


# =============================================================================
# VIDEO PROCESSING
# =============================================================================

def read_video_pyav(container, indices):
    """Decode video frames with PyAV."""
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])


def extract_keyframes_from_video(video_path: str, num_frames: int = 8, temp_dir: str = "./temp_keyframes"):
    """Extract keyframes from video and save temporarily."""
    os.makedirs(temp_dir, exist_ok=True)

    container = av.open(video_path)
    total_frames = container.streams.video[0].frames
    
    if total_frames == 0:
        # Fallback: count frames manually
        total_frames = sum(1 for _ in container.decode(video=0))
        container.seek(0)
    
    indices = np.arange(0, total_frames, max(1, total_frames // num_frames)).astype(int)[:num_frames]
    clip = read_video_pyav(container, indices)

    keyframes = []
    video_name = Path(video_path).stem

    for i, frame in enumerate(clip):
        img = Image.fromarray(frame)
        img_path = os.path.join(temp_dir, f"{video_name}_frame_{i}.png")
        img.save(img_path)
        keyframes.append(img_path)

    container.close()
    return keyframes


# =============================================================================
# INFERENCE
# =============================================================================

def format_mcq_prompt(question: str, options: list, question_type: str) -> str:
    """Format the MCQ prompt with concise traffic analysis instructions."""
    choices_text = "\n".join([f"{i}. {opt}" for i, opt in enumerate(options)])
    num_choices = len(options)
    letter_options = ", ".join([str(i) for i in range(num_choices)])

    return f"""Analyze this dashcam video sequence (frames ordered earliest to latest).

Task Type: {question_type}
- U (Understanding): Identify objects, road features, traffic states
- A (Attribution): Find the root cause of accidents/events
- F (Forecasting): Predict outcomes from trajectories and positions
- R (Reverse): Infer what happened before the current scene
- C (Counterfactual): Reason about "what-if" scenarios
- I (Introspection): Identify preventive actions

Rules:
- Use ONLY visible evidence + traffic rules/physics
- Check ALL frames before concluding absence of objects
- Ground answers in specific visual cues (positions, speeds, distances)

Question: {question}

Choices:
{choices_text}

Respond with ONLY the number ({letter_options}). No explanation.

Answer:""".strip()


def choose_answer(question: str, question_type: str, choices: list, keyframes: list,
                  device: str = "cuda:0", quantization: str = "4bit") -> int:
    """Run inference and return the predicted answer index."""
    tokenizer, model, image_processor = load_model(device=device, quantization=quantization)
    model_device = next(model.parameters()).device

    if quantization in ["4bit", "8bit"]:
        compute_dtype = torch.float16
    else:
        compute_dtype = torch.bfloat16

    # Load and preprocess keyframes
    images = []
    augment = get_dashcam_augmentation()
    for keyframe_path in keyframes:
        try:
            img = Image.open(keyframe_path).convert("RGB")
            # img = augment(img)
            images.append(np.array(img))
        except Exception as e:
            print(f"Warning: Failed to load image {keyframe_path}: {e}")
            continue

    if not images:
        print("Warning: No valid keyframes loaded, returning default answer 0")
        return 0

    video_frames = np.stack(images)
    video = (
        image_processor.preprocess(video_frames, return_tensors="pt")["pixel_values"]
        .to(model_device)
        .bfloat16()
    )
    video = [video]

    # Format prompt
    full_question = format_mcq_prompt(question, choices, question_type)
    
    # Build conversation
    conv_template = "qwen_1_5"
    question_with_token = DEFAULT_IMAGE_TOKEN + f"\n{full_question}"

    conv = copy.deepcopy(conv_templates[conv_template])
    conv.append_message(conv.roles[0], question_with_token)
    conv.append_message(conv.roles[1], None)
    prompt_question = conv.get_prompt()

    # Tokenize
    input_ids = (
        tokenizer_image_token(prompt_question, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        .unsqueeze(0)
        .to(model_device)
    )

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            images=video,
            modalities=["video"],
            do_sample=False,
            temperature=0,
            max_new_tokens=128,
        )

    text_output = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

    # Cleanup
    del video, input_ids, output_ids
    gc.collect()
    torch.cuda.empty_cache()

    # Parse answer
    num_choices = len(choices)
    choice_letters = [str(i) for i in range(num_choices)]
    text_output_upper = text_output.upper()

    if text_output_upper and text_output_upper[0] in choice_letters:
        return choice_letters.index(text_output_upper[0])

    for i, choice_letter in enumerate(choice_letters):
        if choice_letter in text_output_upper[:10]:
            return i

    for i, choice in enumerate(choices):
        if choice.lower() in text_output.lower():
            return i

    print(f"Warning: Could not parse answer from '{text_output}', returning 0")
    return 0


# =============================================================================
# MAIN PROCESSING
# =============================================================================

def main():
    print("=" * 60)
    print("LLaVA-Video SUTD Traffic QA Inference")
    print("=" * 60)
    
    # Load model
    load_model(device=DEVICE, quantization=QUANTIZATION)
    
    # Load questions
    print(f"\nLoading questions from {QUESTIONS_PATH}")
    test_data = []
    with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                if line_num == 0 and row[0] == "record_id":
                    continue
                test_data.append({
                    "record_id": row[0],
                    "vid_id": row[1],
                    "vid_filename": row[2],
                    "perspective": row[3],
                    "question": row[4],
                    "question_type": row[5],
                    "options": [row[6], row[7], row[8], row[9]],
                    "answer": row[10]
                })
            except json.JSONDecodeError as e:
                print(f"Warning: Failed to parse line {line_num}: {e}")
    
    print(f"Total questions: {len(test_data)}")
    
    # Create temp directory for keyframes
    temp_dir = os.path.join(OUTPUT_DIR, "temp_keyframes")
    os.makedirs(temp_dir, exist_ok=True)
    
    # Process each video
    results = []
    correct = 0
    total = 0
    
    for idx, item in enumerate(tqdm(test_data, desc="Processing videos")):
        try:
            vid_filename = item["vid_filename"]
            video_path = os.path.join(VIDEO_DIR, vid_filename)
            
            if not os.path.exists(video_path):
                print(f"Warning: Video not found: {video_path}")
                item["model_answer_idx"] = -1
                item["is_correct"] = False
                results.append(item)
                continue
            
            # Extract keyframes
            keyframes = extract_keyframes_from_video(video_path, NUM_FRAMES, temp_dir)
            
            # Get prediction
            predicted_idx = choose_answer(
                question=item["question"],
                question_type=item["question_type"],
                choices=item["options"],
                keyframes=keyframes,
                device=DEVICE,
                quantization=QUANTIZATION
            )
            
            item["model_answer_idx"] = predicted_idx
            item["is_correct"] = (predicted_idx == item["answer"])
            results.append(item)
            
            total += 1
            if item["is_correct"]:
                correct += 1
            
            # Clean up keyframes
            for kf in keyframes:
                if os.path.exists(kf):
                    os.remove(kf)
            
            # Progress report
            if (idx + 1) % 50 == 0:
                print(f"\nAccuracy at {idx + 1}: {correct}/{total} = {correct/total*100:.2f}%")
            
            # Clear cache
            if (idx + 1) % 10 == 0:
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"\nError processing {item.get('vid_filename', 'unknown')}: {str(e)}")
            import traceback
            traceback.print_exc()
            item["model_answer_idx"] = -1
            item["is_correct"] = False
            results.append(item)
    
    # Save results
    final_accuracy = correct / total * 100 if total > 0 else 0
    
    output_path = os.path.join(OUTPUT_DIR, f"sutd_test_llava_video_{NUM_FRAMES}_frames_{QUANTIZATION}.csv")
    df_results = pd.DataFrame(results)
    df_results = df_results[["record_id", "vid_filename", "model_answer_idx", "answer", "is_correct"]]
    df_results.columns = ["id", "filename", "answer", "gt_answer", "is_correct"]
    df_results.to_csv(output_path, index=False)
    
    # Cleanup temp directory
    if os.path.exists(temp_dir):
        try:
            os.rmdir(temp_dir)
        except:
            pass
    
    print(f"\n{'='*60}")
    print(f"Final Results:")
    print(f"Accuracy: {correct}/{total} = {final_accuracy:.2f}%")
    print(f"VRAM usage: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"Peak VRAM usage: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
    print(f"Results saved to: {output_path}")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

Overwriting llava_video_sutd.py


In [29]:
# If something calls breakpoint(), make it a no-op
%env PYTHONBREAKPOINT=0

# If code is running with "python -m pdb", skip initial stop
%env PYTHONPDBRC=/kaggle/working/pdbrc
!printf "continue\n" > /kaggle/working/pdbrc

!printf "continue\n" > /kaggle/working/.pdbrc
%env HOME=/kaggle/working

env: PYTHONBREAKPOINT=0
env: PYTHONPDBRC=/kaggle/working/pdbrc
env: HOME=/kaggle/working


In [30]:
# Run inference without debugger interruption
!python3 llava_video_sutd.py

Please install pyav to use video processing functions.
VIDEO_DIR: /kaggle/input/sutd-traffic-video-qa/videos_obj_tracking exists: True
QUESTIONS_DIR: /kaggle/input/sutd-traffic-video-qa/questions/questions exists: True
Found 7 question JSONL file(s):
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_all.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_train.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_train_augmented.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_all.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_test.jsonl
 - /kaggle/input/sutd-traffic-video-qa/questions/questions/R3_train.jsonl
QUESTIONS_PATH: /kaggle/input/sutd-traffic-video-qa/questions/questions/R2_test.jsonl
OUTPUT_DIR: /kaggle/working/output
KEYFRAME_DIR: /kaggle/temp/keyframes
LLaVA-Video SUTD Traffic QA Inference
Loading LLaVA-Video model with 

In [ ]:
# Read and analyze results
import pandas as pd

output_path = "/kaggle/working/output/sutd_test_llava_video_8_frames_4bit.csv"
df = pd.read_csv(output_path)

# Display accuracy breakdown
total = len(df)
correct = df['is_correct'].sum()
print(f"Overall Accuracy: {correct}/{total} = {correct/total*100:.2f}%")

# Show accuracy progression
print("\nAccuracy progression:")
running_correct = 0
for i in range(total):
    if df.loc[i, 'is_correct']:
        running_correct += 1
    if (i + 1) % 50 == 0:
        print(f"  At {i + 1}: {running_correct}/{i + 1} = {running_correct/(i+1)*100:.2f}%")

# Display sample results
print("\nSample results:")
print(df.head(10))

In [ ]:
# Download results from vast.ai
from google.colab import files
files.download("/kaggle/working/output/sutd_test_llava_video_8_frames_4bit.csv")